# Notebook 01 (Participant): Train + Generate with EngiOpt CGAN-2D

Goal: implement the core pipeline to produce reproducible artifacts for Notebook 02.


**Before editing:** if you opened from a GitHub URL, click **File -> Save a copy in Drive** first.
That keeps your edits in your own copy and avoids accidental GitHub sync prompts.


In [ ]:
# Colab/local dependency bootstrap
import subprocess
import sys

IN_COLAB = 'google.colab' in sys.modules
FORCE_INSTALL = False  # Set True to force reinstall outside Colab
PACKAGES = ['engibench[beams2d]', 'sqlitedict', 'torch', 'torchvision', 'matplotlib', 'pandas', 'tqdm', 'tyro', 'wandb']
ENGIOPT_GIT = 'git+https://github.com/IDEALLab/EngiOpt.git@codex/dcc26-workshop-notebooks#egg=engiopt'

if IN_COLAB or FORCE_INSTALL:
    print('Installing base dependencies...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *PACKAGES])
    print('Installing EngiOpt from GitHub branch...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', ENGIOPT_GIT])
    print('Dependency install complete.')
else:
    print('Skipping install (using current environment).')


## Part A: Setup

You can complete this notebook without W&B.
If you want remote artifacts, set `USE_WANDB_ARTIFACTS=True` and log in with `wandb.login()`.


### EngiBench vs EngiOpt roles in this notebook

- `Beams2D` (EngiBench): defines dataset fields, conditions, constraints, and simulator-based objective.
- `Generator` (EngiOpt): maps condition vectors + latent noise to candidate designs.
- The **key research question** here is not just reconstruction quality, but whether generated designs remain feasible and competitive under benchmark simulation.


In [ ]:
import json
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch as th
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from engibench.problems.beams2d.v0 import Beams2D

try:
    from engiopt.cgan_2d.cgan_2d import Generator as EngiOptCGAN2DGenerator
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        'Could not import engiopt model class. Run the bootstrap cell first; on Colab, restart runtime after install if needed.'
    ) from exc

USE_WANDB_ARTIFACTS = False
WANDB_PROJECT = 'dcc26-workshop'
WANDB_ENTITY = None
WANDB_ARTIFACT_NAME = 'dcc26_beams2d_generated_artifacts'
WANDB_ARTIFACT_ALIAS = 'latest'
WANDB_LOG_TRAINING = True


def resolve_artifact_dir(create: bool = False) -> Path:
    in_colab = 'google.colab' in sys.modules
    path = Path('/content/dcc26_artifacts') if in_colab else Path('workshops/dcc26/artifacts')
    if create:
        path.mkdir(parents=True, exist_ok=True)
    return path


SEED = 7
random.seed(SEED)
np.random.seed(SEED)
th.manual_seed(SEED)
if th.cuda.is_available():
    th.cuda.manual_seed_all(SEED)

DEVICE = th.device('cuda' if th.cuda.is_available() else 'cpu')
print('device:', DEVICE)

ARTIFACT_DIR = resolve_artifact_dir(create=True)
print('artifact dir:', ARTIFACT_DIR)

CKPT_PATH = ARTIFACT_DIR / 'engiopt_cgan2d_generator_supervised.pt'
HISTORY_PATH = ARTIFACT_DIR / 'training_history.csv'
TRAIN_CURVE_PATH = ARTIFACT_DIR / 'training_curve.png'
LATENT_DIM = 32


In [ ]:
problem = Beams2D(seed=SEED)
train_ds = problem.dataset['train']
test_ds = problem.dataset['test']

condition_keys = problem.conditions_keys
print('condition keys:', condition_keys)

N_TRAIN = 512
subset_idx = np.random.default_rng(SEED).choice(len(train_ds), size=N_TRAIN, replace=False)

conds_np = np.stack([np.array(train_ds[k])[subset_idx].astype(np.float32) for k in condition_keys], axis=1)
designs_np = np.array(train_ds['optimal_design'])[subset_idx].astype(np.float32)
targets_np = (designs_np * 2.0) - 1.0

print('conditions shape:', conds_np.shape)
print('designs shape:', designs_np.shape)
print('target range:', float(targets_np.min()), 'to', float(targets_np.max()))


In [ ]:
# TODO 1: Instantiate model, optimizer, loss, and noise sampler.
# Required objects:
# - model (EngiOptCGAN2DGenerator)
# - optimizer (Adam)
# - criterion (MSELoss)
# - sample_noise(batch_size)

raise NotImplementedError('Complete TODO 1 model setup')


In [ ]:
TRAIN_FROM_SCRATCH = True
EPOCHS = 8
BATCH_SIZE = 64

# TODO 2: Train or load checkpoint.
# Requirements:
# - if TRAIN_FROM_SCRATCH:
#   1) create DataLoader from (conds_np, targets_np)
#   2) run epoch loop and optimize reconstruction loss
#   3) collect train_losses list
#   4) save checkpoint to CKPT_PATH
#   5) save training history CSV to HISTORY_PATH
#   6) save training curve figure to TRAIN_CURVE_PATH
# - elif CKPT_PATH exists: load it
# - else: raise FileNotFoundError

raise NotImplementedError('Complete TODO 2 training/loading')


In [ ]:
# TODO 3: Generate designs and prepare condition records.
# Requirements:
# - sample N_SAMPLES from test dataset
# - run model(sample_noise(...), condition_tensor)
# - map tanh output back to [0, 1]
# - create:
#   gen_designs, baseline_designs, test_conds, conditions_records

raise NotImplementedError('Complete TODO 3 generation')


In [ ]:
# TODO 4: Save Notebook 02 artifacts and (optionally) W&B artifact.
# Required files:
# - generated_designs.npy
# - baseline_designs.npy
# - conditions.json
# Recommended extras:
# - checkpoint (.pt), training_history.csv, training_curve.png

raise NotImplementedError('Complete TODO 4 artifact export')


In [ ]:
# Quick visual side-by-side snapshot
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for i in range(6):
    axes[0, i].imshow(gen_designs[i], cmap='gray', vmin=0, vmax=1)
    axes[0, i].set_title(f'gen {i}')
    axes[0, i].axis('off')

    axes[1, i].imshow(baseline_designs[i], cmap='gray', vmin=0, vmax=1)
    axes[1, i].set_title(f'base {i}')
    axes[1, i].axis('off')

fig.tight_layout()
plt.show()


## Next

Continue with **Notebook 02** to run physics evaluation and benchmark metrics.
